 Main-stick usage: polar 053, Cartesian 059, and ranked play

The plots use the recorded HUMAN port from every `.slp` in the final 96-match
CPU evaluations. A boot can contain two matches; both appear in `match_rows.json`
and both replays are counted. Rollback duplicates are removed and only frames
with frame ID >= 0 are used. This measures executed controller inputs, not
sampled policy logits.

Reconstruction uses a deterministic 64-replay subset of the local
`ranked-anonymized-1/mds-policy-v7/test` split, both players, every stored frame.
It is exploratory: neither MSE nor class usage proves gameplay quality.


Rerun with the project environment and a configured `rclone` `r2:` remote. Missing replays are cached under `/tmp/hal-stick-audit`.


# Selected main-stick candidate: 81 points

Use the [pruned 81-point layout](main_stick_usage/hybrid_81_pruned.png) for the next implementation
and gameplay evaluation. This records the chosen geometry; no existing checkpoint
or runtime codec uses it yet.

- Retain 21 of the original 25 Cartesian core points. Remove only the four
  diagonal points at `(±0.6, ±0.6)` (radius about 0.85). Keep the inner
  diagonals and all four partial cardinal radii: 0.35, 0.50, 0.675, and 0.85.
- Add 12 points at radius 0.75: 30°, 45°, and 60° in each quadrant.
- Use 48 rim points: four exact cardinal points at radius 1.0 and 11 angles
  evenly spaced from 17° through 73° in each quadrant. Keep the exact 1.0
  outputs. Do not move them to 79/80 or add 79/80 classes.

Total: 21 + 12 + 48 = 81. The `candidate(..., keep_outer=False)` code below
defines the exact center order and coordinates. On the disjoint 64-replay
confirmation subset, nearest-center MSE was 0.0002422. This is offline
reconstruction evidence; choose any trained model by gameplay evaluation.


In [ ]:
from __future__ import annotations

import hashlib
import json
import math
import os
import subprocess
import sys
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import peppi_py
import torch
from matplotlib.colors import LogNorm

from hal.data.mds import open_shard
from hal.data.mds import read_shard_index
from hal.data.policy_schema import unpack_stick
from hal.models.controller_codec import STICK_CLUSTER_CENTERS_MAIN
from hal.wire import dedupe_keep_idx

ROOT = Path.cwd()
if not (ROOT / "hal").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "hal").is_dir():
    raise RuntimeError("run this notebook from the HAL repository or its notebooks directory")
CACHE = Path(os.environ.get("HAL_STICK_AUDIT_CACHE", "/tmp/hal-stick-audit"))
DATA = ROOT / "data/processed/ranked-anonymized-1/mds-policy-v7"
N_REAL_REPLAYS = 64
CHUNK = 16_384

SOURCES = {
    "polar 053": {
        "run_id": "yi7yyyu0",
        "r2": "r2:hal/runs/260910-004344_053_polar_controller_polar053-d256-L16-h4-Lc256-t128x4-o1-2-3-4-5-6-9-12-16-20-d2r2-fixed-polar-sticks-nonlinear-head-trunk-skip-projectiles-v8-all-adamw-alr0.0017-awd0.0001-awr-v-near-b199.5-g0.99618-wu512__o53-polar/checkpoints/eval96-step-0016384",
        "local": CACHE / "polar",
        "checkpoint_sha256": "305c0776040a74c75b2ea5eaa5d5922efb938e1bd0e0a87ebc6f115f793a77e5",
    },
    "059": {
        "run_id": "vywk3cih",
        "r2": "r2:hal/runs/260921-092548_059_muon_history_decoder_o59v5-cooldown32768/checkpoints/eval96-step-0131072",
        "local": CACHE / "o59",
        "checkpoint_sha256": "52b5233ed506f59f514f7e90a6a6111206152db7413f451dc35be5c30d1e671b",
    },
}


def require_xy(xy: np.ndarray, where: str) -> np.ndarray:
    if xy.ndim != 2 or xy.shape[1] != 2 or not np.isfinite(xy).all():
        raise ValueError(f"{where}: expected finite [frames, 2] stick values")
    if np.max(np.abs(xy)) > 1.00001:
        raise ValueError(f"{where}: stick value outside [-1, 1]")
    return xy.astype(np.float32, copy=False)


def eval_inputs(label: str, source: dict[str, object]) -> tuple[np.ndarray, dict[str, object]]:
    local = source["local"]
    assert isinstance(local, Path)
    if not (local / "match_rows.json").is_file():
        subprocess.run(["rclone", "copy", str(source["r2"]), str(local), "--transfers", "8"], check=True)
    evidence = json.loads((local / "match_rows.json").read_text())
    protocol = evidence["protocol"]
    if (
        protocol["checkpoint_sha256"] != source["checkpoint_sha256"]
        or protocol["n_matchups"] != 96
        or protocol["cpu_level"] != 9
        or protocol["ego_port"] != 1
    ):
        raise ValueError(f"{label}: wrong checkpoint or CPU evaluation protocol")
    paths = sorted(local.glob("boot_*/*.slp"))
    if len(paths) != len(evidence["rows"]):
        raise ValueError(f"{label}: {len(paths)} replays for {len(evidence['rows'])} match rows")
    boot_counts: dict[int, int] = {}
    for row in evidence["rows"]:
        boot = int(row["boot_index"])
        boot_counts[boot] = boot_counts.get(boot, 0) + 1
    if set(boot_counts) != set(range(96)):
        raise ValueError(f"{label}: incomplete boot set")
    for boot, count in boot_counts.items():
        if len(list(local.glob(f"boot_{boot:03d}/*.slp"))) != count:
            raise ValueError(f"{label}: boot {boot} replay count differs from match rows")

    digest = hashlib.sha256()
    for path in paths:
        digest.update(str(path.relative_to(local)).encode())
        digest.update(b"\0")
        digest.update(hashlib.sha256(path.read_bytes()).digest())

    arrays = []
    for path in paths:
        game = peppi_py.read_slippi(str(path), skip_frames=False)
        if game.frames is None or game.frames.id is None:
            raise ValueError(f"{path}: missing frames")
        human = [i for i, player in enumerate(game.start.players) if int(player.type.value) == 0]
        cpu = [i for i, player in enumerate(game.start.players) if int(player.type.value) == 1]
        if len(human) != 1 or len(cpu) != 1:
            raise ValueError(f"{path}: expected one human policy port and one CPU port")
        pre = game.frames.ports[human[0]].leader.pre
        if pre is None or pre.joystick is None:
            raise ValueError(f"{path}: missing policy controller inputs")
        ids = np.asarray(game.frames.id.to_pylist(), dtype=np.int32)
        xy = np.stack((pre.joystick.x.to_numpy(), pre.joystick.y.to_numpy()), axis=-1)
        keep = dedupe_keep_idx(ids)
        keep = keep[ids[keep] >= 0]
        arrays.append(require_xy(xy[keep], str(path)))
    return np.concatenate(arrays), {
        "run_id": source["run_id"],
        "checkpoint_sha256": protocol["checkpoint_sha256"],
        "match_rows_sha256": hashlib.sha256((local / "match_rows.json").read_bytes()).hexdigest(),
        "replay_manifest_sha256": digest.hexdigest(),
        "protocol": protocol,
        "replays": len(paths),
        "boots": len(boot_counts),
        "frames": sum(len(a) for a in arrays),
    }


eval_xy = {}
eval_metadata = {}
for label, source in SOURCES.items():
    eval_xy[label], eval_metadata[label] = eval_inputs(label, source)
print(json.dumps(eval_metadata, indent=2))
print("git_sha:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())
print(
    "environment:",
    {
        "python": sys.version.split()[0],
        "numpy": np.__version__,
        "torch": torch.__version__,
        "matplotlib": matplotlib.__version__,
    },
)

In [ ]:
def ranked_test_inputs() -> tuple[np.ndarray, dict[str, object]]:
    index = DATA / "test/index.json"
    if not index.is_file():
        raise FileNotFoundError(index)
    info = read_shard_index(DATA, "test")
    total = sum(int(shard["samples"]) for shard in info)
    selected = np.linspace(0, total - 1, N_REAL_REPLAYS, dtype=np.int64)
    if len(np.unique(selected)) != N_REAL_REPLAYS:
        raise ValueError("ranked test split too small for the selected subset")
    arrays = []
    offset = 0
    for shard in info:
        count = int(shard["samples"])
        rows = selected[(selected >= offset) & (selected < offset + count)] - offset
        if len(rows):
            with open_shard(DATA, "test", shard, CACHE) as reader:
                for row_id in rows:
                    row = reader[int(row_id)]
                    length = int(row["num_frames"])
                    for port in ("p1", "p2"):
                        x = unpack_stick(row[f"{port}_main_stick_x"])
                        y = unpack_stick(row[f"{port}_main_stick_y"])
                        if len(x) != length or len(y) != length:
                            raise ValueError(f"shard row {int(row_id)}: stick length mismatch")
                        arrays.append(require_xy(np.stack((x, y), axis=-1), f"ranked row {int(row_id)}"))
        offset += count
    if len(arrays) != 2 * N_REAL_REPLAYS:
        raise ValueError("ranked test subset incomplete")
    return np.concatenate(arrays), {
        "split": "ranked-anonymized-1/mds-policy-v7/test",
        "index_sha256": hashlib.sha256(index.read_bytes()).hexdigest(),
        "row_ids": selected.tolist(),
        "replays": N_REAL_REPLAYS,
        "frames": sum(len(a) for a in arrays),
    }


real_xy, real_metadata = ranked_test_inputs()
print({key: value for key, value in real_metadata.items() if key != "row_ids"})
print("row_ids:", real_metadata["row_ids"])

# Executed actions

A common 2D histogram scale makes the two policy densities comparable.
The white markers show the available class positions; their color in the
second figure shows actual occupancy after running each codec's quantizer.
The third compares policy and ranked shares on each codec's own classes.
These are different populations, so that ratio is descriptive, not a
calibrated measure of which actions a policy should take.



In [ ]:
cartesian_centers = STICK_CLUSTER_CENTERS_MAIN.numpy()


def polar_centers(angle_counts: tuple[int, int, int]) -> np.ndarray:
    radii = np.exp(np.linspace(np.log(23), np.log(80), 3, dtype=np.float64)) / 80
    points = [(0.0, 0.0)]
    for radius, count in zip(radii, angle_counts, strict=True):
        for angle in np.arange(count) * (2 * np.pi / count) - np.pi:
            points.append((radius * np.cos(angle), radius * np.sin(angle)))
    return np.asarray(points, dtype=np.float32)


POLAR85 = (4, 16, 64)
POLAR65 = (4, 12, 48)
HYBRID73 = np.concatenate((cartesian_centers[:25], polar_centers((4, 16, 48))[-48:]))
centers = {"polar 053": polar_centers(POLAR85), "059": cartesian_centers}

bins = np.linspace(-1, 1, 82)
histograms = {label: np.histogram2d(xy[:, 0], xy[:, 1], bins=(bins, bins))[0].T for label, xy in eval_xy.items()}
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5), sharex=True, sharey=True, layout="constrained")
scale = LogNorm(vmin=1, vmax=max(float(h.max()) for h in histograms.values()))
for ax, (label, hist) in zip(axes, histograms.items(), strict=True):
    image = ax.pcolormesh(bins, bins, np.ma.masked_equal(hist, 0), norm=scale, cmap="magma", shading="auto")
    ax.scatter(centers[label][:, 0], centers[label][:, 1], s=13, facecolors="none", edgecolors="cyan", linewidths=0.65)
    ax.set(
        title=f"{label}: {len(eval_xy[label]):,} policy frames",
        xlabel="main-stick x",
        ylabel="main-stick y",
        aspect="equal",
    )
fig.colorbar(image, ax=axes, label="recorded frames per spatial bin (log)")
plt.show()

In [ ]:
def polar_indices(xy: np.ndarray, angle_counts: tuple[int, int, int]) -> np.ndarray:
    raw = torch.round(torch.from_numpy(xy) * 80)
    radius = raw.square().sum(-1).sqrt()
    log_fraction = (torch.log(radius + 1e-3) - math.log(23)) / (math.log(80) - math.log(23))
    ring = torch.round(log_fraction * 2).long().clamp(0, 2)
    ring = torch.where(radius <= 22, torch.zeros_like(ring), ring + 1)
    counts = torch.tensor((1, *angle_counts), dtype=torch.long)[ring]
    angle = (torch.atan2(raw[:, 1], raw[:, 0]) + math.pi) / (2 * math.pi)
    bucket = torch.round(angle * counts).long().remainder(counts)
    offsets = torch.tensor((0, 1, 1 + angle_counts[0], 1 + angle_counts[0] + angle_counts[1]))
    return (offsets[ring] + bucket).numpy()


def nearest_indices(xy: np.ndarray, codebook: np.ndarray) -> np.ndarray:
    result = np.empty(len(xy), dtype=np.int32)
    reference = torch.from_numpy(codebook)
    for start in range(0, len(xy), CHUNK):
        block = torch.from_numpy(xy[start : start + CHUNK])
        result[start : start + CHUNK] = (block[:, None] - reference).square().sum(-1).argmin(-1).numpy()
    return result


def quantize(xy: np.ndarray, name: str) -> tuple[np.ndarray, np.ndarray]:
    if name in ("059", "hybrid 73"):
        codebook = cartesian_centers if name == "059" else HYBRID73
        index = nearest_indices(xy, codebook)
    else:
        counts = {"polar 053": POLAR85, "polar 65": POLAR65}[name]
        codebook = polar_centers(counts)
        index = polar_indices(xy, counts)
    return index, codebook


def occupancy(xy: np.ndarray, name: str) -> np.ndarray:
    index, codebook = quantize(xy, name)
    return np.bincount(index, minlength=len(codebook))


occupancies = {label: occupancy(eval_xy[label], label) for label in eval_xy}
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5), sharex=True, sharey=True, layout="constrained")
max_count = max(int(v.max()) for v in occupancies.values())
for ax, label in zip(axes, occupancies, strict=True):
    counts = occupancies[label]
    points = centers[label]
    used = counts > 0
    ax.scatter(
        points[used, 0],
        points[used, 1],
        c=counts[used],
        s=95,
        cmap="viridis",
        norm=LogNorm(vmin=1, vmax=max_count),
        edgecolors="black",
        linewidths=0.35,
    )
    ax.scatter(points[~used, 0], points[~used, 1], marker="x", s=50, c="lightgray", linewidths=1.2)
    ax.set(
        title=f"{label}: {sum(counts == 0)} unused / {len(counts)} classes",
        xlabel="main-stick x",
        ylabel="main-stick y",
        aspect="equal",
    )
fig.colorbar(axes[0].collections[0], ax=axes, label="assigned policy frames per class (log)")
plt.show()

for label, counts in occupancies.items():
    share = counts / counts.sum()
    top = np.argsort(-counts)[:8]
    low = np.argsort(counts)[:8]
    print(
        f"\n{label}: neutral {share[0]:.1%}; unused {sum(counts == 0)}; <0.1% {sum(share < 0.001)}; effective classes {1 / np.sum(share**2):.1f}"
    )
    print("most used:", [(int(i), tuple(np.round(centers[label][i], 3)), round(float(share[i]), 4)) for i in top])
    print("least used:", [(int(i), tuple(np.round(centers[label][i], 3)), round(float(share[i]), 6)) for i in low])

real_occupancies = {label: occupancy(real_xy, label) for label in eval_xy}
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5), sharex=True, sharey=True, layout="constrained")
for ax, label in zip(axes, eval_xy, strict=True):
    real_share = real_occupancies[label] / len(real_xy)
    policy_share = occupancies[label] / len(eval_xy[label])
    supported = real_share >= 0.001
    ratio = np.log2(np.maximum(policy_share[supported], 1 / len(eval_xy[label])) / real_share[supported])
    points = centers[label]
    image = ax.scatter(
        points[supported, 0],
        points[supported, 1],
        c=ratio,
        s=100,
        cmap="coolwarm",
        vmin=-4,
        vmax=4,
        edgecolors="black",
        linewidths=0.35,
    )
    ax.scatter(points[~supported, 0], points[~supported, 1], marker="x", s=50, c="lightgray", linewidths=1.2)
    ax.set(title=f"{label}: policy / ranked class share", xlabel="main-stick x", ylabel="main-stick y", aspect="equal")
    misses = np.flatnonzero(supported & (policy_share < real_share / 4))
    excess = np.flatnonzero(supported & (policy_share > real_share * 4))
    print(f"{label}: real-supported classes >4x underused: {misses.tolist()}; >4x overused: {excess.tolist()}")
fig.colorbar(image, ax=axes, label="log2(policy share / ranked share); gray: <0.1% ranked share")
plt.show()

# Reconstruction on real ranked play

MSE is the mean of squared x and y errors, per scalar coordinate. The codec
rows use their actual quantization rule. Polar 65 keeps polar's radii and
reallocates angles at the same class count as 059. Hybrid 73 keeps all 25
Cartesian inner/mid centers and uses a denser 48-spoke polar rim.



In [ ]:
REGIONS = {
    "neutral": np.linalg.norm(real_xy, axis=1) == 0,
    "inner (<0.4)": (np.linalg.norm(real_xy, axis=1) > 0) & (np.linalg.norm(real_xy, axis=1) < 0.4),
    "mid (0.4–0.95)": (np.linalg.norm(real_xy, axis=1) >= 0.4) & (np.linalg.norm(real_xy, axis=1) < 0.95),
    "rim (≥0.95)": np.linalg.norm(real_xy, axis=1) >= 0.95,
}
names = ("059", "polar 053", "polar 65", "hybrid 73")
result = {}
mse_frames = {}
for name in names:
    index, codebook = quantize(real_xy, name)
    mse_frame = np.square(real_xy - codebook[index]).mean(axis=1)
    mse_frames[name] = mse_frame
    result[name] = {"classes": len(codebook), "all": float(mse_frame.mean())}
    result[name].update({region: float(mse_frame[mask].mean()) for region, mask in REGIONS.items()})
    result[name]["occupancy"] = np.bincount(index, minlength=len(codebook))

print("Frame-weighted MSE per stick coordinate on ranked-anon-1 test subset")
print(f"{'layout':<16} {'classes':>7} {'all':>10} {'neutral':>10} {'inner':>10} {'mid':>10} {'rim':>10}")
for name in names:
    row = result[name]
    print(
        f"{name:<16} {row['classes']:>7} {row['all']:>10.6f} {row['neutral']:>10.6f} {row['inner (<0.4)']:>10.6f} {row['mid (0.4–0.95)']:>10.6f} {row['rim (≥0.95)']:>10.6f}"
    )
print("real-data region shares:", {key: round(float(mask.mean()), 4) for key, mask in REGIONS.items()})

fig, ax = plt.subplots(figsize=(9, 4.5), layout="constrained")
region_names = ("inner (<0.4)", "mid (0.4–0.95)", "rim (≥0.95)")
x = np.arange(len(region_names))
width = 0.2
for j, name in enumerate(names):
    ax.bar(
        x + (j - 1.5) * width,
        [result[name][region] for region in region_names],
        width,
        label=f"{name} ({result[name]['classes']})",
    )
ax.set(
    xticks=x, xticklabels=region_names, ylabel="MSE per coordinate", title="Ranked test reconstruction by stick radius"
)
ax.legend()
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(16, 4.7), sharex=True, sharey=True, layout="constrained")
comparison = ("059", "polar 053", "hybrid 73")
bin_counts = np.histogram2d(real_xy[:, 0], real_xy[:, 1], bins=(bins, bins))[0].T
error_maps = {}
for name in comparison:
    weighted = np.histogram2d(real_xy[:, 0], real_xy[:, 1], bins=(bins, bins), weights=mse_frames[name])[0].T
    error_maps[name] = np.divide(weighted, bin_counts, out=np.full_like(weighted, np.nan), where=bin_counts >= 100)
maximum = max(float(np.nanpercentile(value, 99)) for value in error_maps.values())
for ax, name in zip(axes, comparison, strict=True):
    image = ax.pcolormesh(bins, bins, error_maps[name], vmin=0, vmax=maximum, cmap="inferno", shading="auto")
    ax.set(title=name, xlabel="main-stick x", ylabel="main-stick y", aspect="equal")
fig.colorbar(image, ax=axes, label="mean squared error per coordinate (bins with ≥100 frames)")
plt.show()

The policy heatmaps show where each trained agent actually moved the stick;
their distributions need not match each other or ranked play. The MSE table
isolates codec geometry on the **same** real inputs. A lower reconstruction
MSE is a reason to test a layout in a new training run, not evidence that it
improves live play. Existing checkpoints cannot use a changed vocabulary.
On this fixed sample, hybrid 73 lowers overall reconstruction MSE to about
0.000407 from 059's 0.000440; polar 053 has lower rim error but much larger
mid-radius error. The hybrid is an experiment candidate, not a model result.


# Axis-aware hybrid

The ranked test subset has no nonzero rim component smaller than 23/80. Its nearest
noncardinal rim angle is 16.84° from an axis. The original hybrid has 16 rim centers
inside those gaps. Eight receive no ranked assignments. The other eight still
represent 19,148 reachable frames, so deleting all 16 increases MSE.

The treatment keeps the same 25 inner and middle centers and places the rim's
noncardinal points between 17° and 73° in each quadrant. Four exact cardinal
points remain. The control is the original evenly spaced hybrid. Both use
nearest-center assignment on the same 64 ranked test replays; the 65-point row
shows the class-count tradeoff. These are offline codec measurements, not
gameplay evaluations.


In [ ]:
from matplotlib.patches import Wedge


def axis_aware_centers(per_quadrant: int) -> np.ndarray:
    quadrant = np.linspace(np.deg2rad(17), np.deg2rad(73), per_quadrant)
    angles = np.concatenate((np.deg2rad(np.arange(4) * 90), *[quadrant + turn * np.pi / 2 for turn in range(4)]))
    rim = np.stack((np.cos(angles), np.sin(angles)), axis=-1).astype(np.float32)
    return np.concatenate((cartesian_centers[:25], rim))


axis_65 = axis_aware_centers(9)
axis_73 = axis_aware_centers(11)
old_rim = HYBRID73[25:]
rim_angle = np.mod(np.arctan2(old_rim[:, 1], old_rim[:, 0]), np.pi / 2)
axis_distance = np.minimum(rim_angle, np.pi / 2 - rim_angle)
gap_edge = np.arcsin(23 / 80)
in_gap = (axis_distance > 1e-6) & (axis_distance < gap_edge)
old_assignments = np.bincount(nearest_indices(real_xy, HYBRID73), minlength=len(HYBRID73))[25:]

real_radius = np.linalg.norm(real_xy, axis=1)
real_rim = real_xy[real_radius >= 0.95]
rim_components = np.abs(real_rim).ravel()
rim_components = rim_components[rim_components > 0]
real_angle = np.mod(np.arctan2(real_rim[:, 1], real_rim[:, 0]), np.pi / 2)
real_axis_distance = np.minimum(real_angle, np.pi / 2 - real_angle)
noncardinal = real_axis_distance[real_axis_distance > 1e-6]
print(
    f"Smallest nonzero rim component: {rim_components.min():.4f}; nearest noncardinal angle: {np.rad2deg(noncardinal.min()):.2f}°"
)
print(
    f"Original 73: {in_gap.sum()} rim centers inside axis gaps; "
    f"{np.sum(old_assignments[in_gap] == 0)} receive no ranked assignments; "
    f"the others receive {old_assignments[in_gap].sum():,} frames"
)

layouts = {
    "original 73": HYBRID73,
    "delete gap → 57": np.concatenate((cartesian_centers[:25], old_rim[~in_gap])),
    "axis-aware 65": axis_65,
    "axis-aware 73": axis_73,
}
print(f"{'layout':<18} {'classes':>7} {'overall MSE':>12} {'rim MSE':>12}")
for label, codebook in layouts.items():
    assigned = nearest_indices(real_xy, codebook)
    error = np.square(real_xy - codebook[assigned]).mean(axis=1)
    print(f"{label:<18} {len(codebook):>7} {error.mean():>12.6f} {error[real_radius >= 0.95].mean():>12.6f}")

fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharex=True, sharey=True, layout="constrained")
for ax, (label, codebook) in zip(
    axes,
    (("original 73", HYBRID73), ("axis-aware 65", axis_65), ("axis-aware 73", axis_73)),
    strict=True,
):
    for direction in range(0, 360, 90):
        ax.add_patch(
            Wedge(
                (0, 0),
                1.07,
                direction - np.rad2deg(gap_edge),
                direction + np.rad2deg(gap_edge),
                width=0.15,
                facecolor="#dce1e6",
                edgecolor="none",
                zorder=0,
            )
        )
    ax.add_patch(plt.Circle((0, 0), 1, fill=False, color="#b8c0c8", lw=0.8))
    ax.scatter(codebook[:25, 0], codebook[:25, 1], s=46, color="#2878a8", label="inner + middle")
    if label == "original 73":
        ax.scatter(old_rim[~in_gap, 0], old_rim[~in_gap, 1], s=44, color="#dc6b4b", label="rim")
        ax.scatter(
            old_rim[in_gap, 0], old_rim[in_gap, 1], s=62, marker="x", color="#89929d", label="center inside gap"
        )
    else:
        ax.scatter(codebook[25:, 0], codebook[25:, 1], s=44, color="#dc6b4b", label="rim")
    ax.set(
        title=label, xlabel="main-stick x", ylabel="main-stick y", aspect="equal", xlim=(-1.1, 1.1), ylim=(-1.1, 1.1)
    )
axes[0].legend(loc="upper left", fontsize=8)
fig.suptitle("Rim centers moved out of the Melee axis gaps")
fig.savefig(ROOT / "notebooks/main_stick_usage/hybrid_73_axis_aware.png", dpi=175, bbox_inches="tight")
plt.show()

# Moving rim capacity into the quadrant middles

The 25-point Cartesian core has two diagonal radii per quadrant but no point
between the cardinal and diagonal directions at middle radius. This comparison
holds the codebook at 73 classes. The control uses 48 axis-aware rim points.
The treatments move 8 or 16 of those rim points to the quadrant interiors.
All rows use nearest-center assignment on the same 64 ranked test replays.

The strongest tested allocation adds two points per quadrant at 30° and 60°,
radius 0.65. It retains 40 rim points. This is a codec reconstruction result,
not a gameplay evaluation or an optimized codebook.


In [ ]:
def middle_spokes(radius: float) -> np.ndarray:
    angles = np.deg2rad([turn * 90 + angle for turn in range(4) for angle in (30, 60)])
    return (radius * np.stack((np.cos(angles), np.sin(angles)), axis=-1)).astype(np.float32)


middle_65 = middle_spokes(0.65)
middle_85 = middle_spokes(0.85)
middle_focused_73 = np.concatenate((cartesian_centers[:25], middle_65, axis_65[25:]))
outer_middle_73 = np.concatenate((cartesian_centers[:25], middle_85, axis_65[25:]))
two_middle_rings_73 = np.concatenate((cartesian_centers[:25], middle_65, middle_85, axis_aware_centers(7)[25:]))

middle_mask = (real_radius >= 0.4) & (real_radius < 0.95)
rim_mask = real_radius >= 0.95
middle_layouts = {
    "rim-focused 73": axis_73,
    "8 middle at r=.65": middle_focused_73,
    "8 middle at r=.85": outer_middle_73,
    "16 middle, 32 rim": two_middle_rings_73,
}
print(f"{'layout':<22} {'classes':>7} {'overall MSE':>12} {'middle MSE':>12} {'rim MSE':>12}")
for label, codebook in middle_layouts.items():
    assigned = nearest_indices(real_xy, codebook)
    error = np.square(real_xy - codebook[assigned]).mean(axis=1)
    print(
        f"{label:<22} {len(codebook):>7} {error.mean():>12.6f} "
        f"{error[middle_mask].mean():>12.6f} {error[rim_mask].mean():>12.6f}"
    )

fig, axes = plt.subplots(1, 2, figsize=(10.7, 5.4), sharex=True, sharey=True, layout="constrained")
for ax, (title, codebook, middle_count) in zip(
    axes,
    (
        ("48 rim · no new middle points", axis_73, 0),
        ("40 rim · 8 new middle points", middle_focused_73, 8),
    ),
    strict=True,
):
    for direction in range(0, 360, 90):
        ax.add_patch(
            Wedge(
                (0, 0),
                1.07,
                direction - np.rad2deg(gap_edge),
                direction + np.rad2deg(gap_edge),
                width=0.15,
                facecolor="#e7eaee",
                edgecolor="none",
                zorder=0,
            )
        )
    ax.add_patch(plt.Circle((0, 0), 1, fill=False, color="#aab4be", lw=0.8))
    ax.axhline(0, color="#d9dfe5", lw=0.6, zorder=0)
    ax.axvline(0, color="#d9dfe5", lw=0.6, zorder=0)
    ax.scatter(codebook[:25, 0], codebook[:25, 1], s=48, color="#2878a8", label="existing core")
    if middle_count:
        ax.scatter(
            codebook[25 : 25 + middle_count, 0],
            codebook[25 : 25 + middle_count, 1],
            s=68,
            color="#61943b",
            label="new quadrant points",
        )
    ax.scatter(
        codebook[25 + middle_count :, 0],
        codebook[25 + middle_count :, 1],
        s=44,
        color="#dc6b4b",
        label="rim",
    )
    ax.set(
        title=title, xlabel="main-stick x", ylabel="main-stick y", aspect="equal", xlim=(-1.1, 1.1), ylim=(-1.1, 1.1)
    )
handles, labels = axes[1].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=3, bbox_to_anchor=(0.5, -0.12))
fig.suptitle("Two 73-point layouts · gray arcs mark the axis gaps")
fig.savefig(ROOT / "notebooks/main_stick_usage/hybrid_73_middle_aware.png", dpi=175, bbox_inches="tight")
plt.show()

# Are the two 45° radii useful?

The earlier 73-class comparison held the class count fixed and used only the
original 64 ranked test replays. Here the treatment adds 30° and 60° middle
points at radius 0.75; controls remove the existing 45° points at radii about
0.50 and 0.85, separately and together. The rim is held fixed for the ablations.
We also compare larger class counts. The original 64 replays were used for the
initial search; 64 disjoint midpoint replays checked candidate radii; 64 more
disjoint quarter-point replays confirm the short list. All replays are from the
same ranked-anonymized-1 test split. Each metric weights recorded frames, not
unique stick positions. This is exploratory offline codec evidence. Select a
training layout by gameplay evaluation.


In [ ]:
def ranked_rows(ids: np.ndarray) -> np.ndarray:
    arrays = []
    offset = 0
    for shard in read_shard_index(DATA, "test"):
        count = int(shard["samples"])
        rows = ids[(ids >= offset) & (ids < offset + count)] - offset
        if len(rows):
            with open_shard(DATA, "test", shard, CACHE) as reader:
                for row_id in rows:
                    row = reader[int(row_id)]
                    length = int(row["num_frames"])
                    for port in ("p1", "p2"):
                        x = unpack_stick(row[f"{port}_main_stick_x"])
                        y = unpack_stick(row[f"{port}_main_stick_y"])
                        if len(x) != length or len(y) != length:
                            raise ValueError(f"shard row {int(row_id)}: stick length mismatch")
                        arrays.append(require_xy(np.stack((x, y), axis=-1), f"ranked row {int(row_id)}"))
        offset += count
    if len(arrays) != 2 * len(ids):
        raise ValueError("ranked subset incomplete")
    return np.concatenate(arrays)


original_ids = np.asarray(real_metadata["row_ids"], dtype=np.int64)
midpoint_ids = np.concatenate(((original_ids[:-1] + original_ids[1:]) // 2, [original_ids[-1] - 1]))
quarter_ids = np.concatenate((original_ids[:-1] + (original_ids[1:] - original_ids[:-1]) // 4, [original_ids[-1] - 4]))
if any(
    np.intersect1d(left, right).size
    for left, right in ((original_ids, midpoint_ids), (original_ids, quarter_ids), (midpoint_ids, quarter_ids))
):
    raise ValueError("ranked subsets overlap")
samples = {
    "original": real_xy,
    "midpoint": ranked_rows(midpoint_ids),
    "confirmation": ranked_rows(quarter_ids),
}
print(
    "ranked replay counts:",
    {
        key: len(ids)
        for key, ids in (("original", original_ids), ("midpoint", midpoint_ids), ("confirmation", quarter_ids))
    },
)
print("ranked frame counts:", {key: len(xy) for key, xy in samples.items()})
print("midpoint row ids:", midpoint_ids.tolist())
print("confirmation row ids:", quarter_ids.tolist())


def middle_points(angles: tuple[float, ...], radii: tuple[float, ...]) -> np.ndarray:
    points = []
    for radius in radii:
        for turn in range(4):
            for angle in angles:
                theta = np.deg2rad(turn * 90 + angle)
                points.append((radius * np.cos(theta), radius * np.sin(theta)))
    return np.asarray(points, dtype=np.float32).reshape(-1, 2)


inner_diagonal = np.array([10, 12, 14, 16])
outer_diagonal = np.array([18, 20, 22, 24])


def candidate(
    *,
    angles: tuple[float, ...] = (),
    radii: tuple[float, ...] = (),
    rim_count: int = 40,
    keep_inner: bool = True,
    keep_outer: bool = True,
) -> np.ndarray:
    dropped = []
    if not keep_inner:
        dropped.extend(inner_diagonal)
    if not keep_outer:
        dropped.extend(outer_diagonal)
    core = np.delete(cartesian_centers[:25], dropped, axis=0)
    rim = {40: axis_65[25:], 48: axis_73[25:]}[rim_count]
    return np.concatenate((core, middle_points(angles, radii), rim))


def weighted_error(xy: np.ndarray, centers: np.ndarray) -> tuple[float, float, float]:
    points, weights = np.unique(xy, axis=0, return_counts=True)
    squared = np.square(points[:, None, :] - centers[None, :, :]).sum(axis=2)
    per_point = squared.min(axis=1) / 2
    radius = np.linalg.norm(points, axis=1)
    middle = (radius >= 0.4) & (radius < 0.95)
    rim = radius >= 0.95
    return (
        float(np.average(per_point, weights=weights)),
        float(np.average(per_point[middle], weights=weights[middle])),
        float(np.average(per_point[rim], weights=weights[rim])),
    )


middle73 = candidate(angles=(30, 60), radii=(0.75,))
middle85 = candidate(angles=(30, 45, 60), radii=(0.75,), rim_count=48)
shortlist = {
    "core + 40 rim": candidate(),
    "remove outer 45°; add 30°/60°": candidate(angles=(30, 60), radii=(0.75,), keep_outer=False),
    "keep both 45°; add 30°/60°": middle73,
    "keep both; add 30°/45°/60°": candidate(angles=(30, 45, 60), radii=(0.75,)),
    "keep both; add 30°/60°; 48 rim": candidate(angles=(30, 60), radii=(0.75,), rim_count=48),
    "keep both; add 30°/45°/60°; 48 rim": middle85,
}
print(f"{'layout':<39} {'N':>3} {'original':>10} {'midpoint':>10} {'confirm':>10}")
for label, centers in shortlist.items():
    errors = [weighted_error(xy, centers)[0] for xy in samples.values()]
    print(f"{label:<39} {len(centers):>3} {errors[0]:>10.7f} {errors[1]:>10.7f} {errors[2]:>10.7f}")

ablations = {
    "keep both 45°": middle73,
    "remove inner 45°": candidate(angles=(30, 60), radii=(0.75,), keep_inner=False),
    "remove outer 45°": candidate(angles=(30, 60), radii=(0.75,), keep_outer=False),
    "remove both 45°": candidate(angles=(30, 60), radii=(0.75,), keep_inner=False, keep_outer=False),
    "replace both with 30°/60° at .55 and .75": candidate(
        angles=(30, 60), radii=(0.55, 0.75), keep_inner=False, keep_outer=False
    ),
}
print(f"{'45° ablation':<42} {'N':>3} {'confirmation MSE':>17} {'middle MSE':>12}")
for label, centers in ablations.items():
    overall, middle, _ = weighted_error(samples["confirmation"], centers)
    print(f"{label:<42} {len(centers):>3} {overall:>17.7f} {middle:>12.7f}")

fig, axes = plt.subplots(1, 2, figsize=(10.7, 5.4), sharex=True, sharey=True, layout="constrained")
for ax, (title, codebook, extra_count) in zip(
    axes,
    (("73 points: 30°/60° at r=.75", middle73, 8), ("85 points: add 45° + denser rim", middle85, 12)),
    strict=True,
):
    for direction in range(0, 360, 90):
        ax.add_patch(
            Wedge(
                (0, 0),
                1.07,
                direction - np.rad2deg(gap_edge),
                direction + np.rad2deg(gap_edge),
                width=0.15,
                facecolor="#e7eaee",
                edgecolor="none",
                zorder=0,
            )
        )
    ax.add_patch(plt.Circle((0, 0), 1, fill=False, color="#aab4be", lw=0.8))
    ax.axhline(0, color="#d9dfe5", lw=0.6, zorder=0)
    ax.axvline(0, color="#d9dfe5", lw=0.6, zorder=0)
    ax.scatter(codebook[:25, 0], codebook[:25, 1], s=48, color="#2878a8", label="existing core")
    ax.scatter(
        codebook[25 : 25 + extra_count, 0],
        codebook[25 : 25 + extra_count, 1],
        s=68,
        color="#61943b",
        label="new quadrant points",
    )
    ax.scatter(codebook[25 + extra_count :, 0], codebook[25 + extra_count :, 1], s=44, color="#dc6b4b", label="rim")
    ax.set(
        title=title, xlabel="main-stick x", ylabel="main-stick y", aspect="equal", xlim=(-1.1, 1.1), ylim=(-1.1, 1.1)
    )
handles, labels = axes[1].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=3, bbox_to_anchor=(0.5, -0.12))
fig.suptitle("Keep the existing diagonal radii; add middle spokes")
fig.savefig(ROOT / "notebooks/main_stick_usage/hybrid_quadrant_candidates.png", dpi=175, bbox_inches="tight")
plt.show()

# Prune the nearly redundant outer 45° points

The 85-point candidate has three middle 45° radii: about 0.50, 0.75, and 0.85.
Removing the four old radius-0.85 diagonal centers leaves 81 classes and changes
confirmation MSE by less than 0.000004. At the same 81-class budget, this is
better than dropping the new radius-0.75 diagonal centers. Removing the inner
radius-0.50 diagonal centers has a much larger cost. These ablations keep the
same 48 axis-aware rim centers and the same ranked replay subsets.


In [ ]:
pruned81 = candidate(angles=(30, 45, 60), radii=(0.75,), rim_count=48, keep_outer=False)
old81 = candidate(angles=(30, 60), radii=(0.75,), rim_count=48)
no_inner81 = candidate(angles=(30, 45, 60), radii=(0.75,), rim_count=48, keep_inner=False)
pruning = {
    "85: all three 45° radii": middle85,
    "81: remove old 45° at r≈.85": pruned81,
    "81: remove new 45° at r=.75": old81,
    "81: remove old 45° at r≈.50": no_inner81,
}
print(f"{'layout':<34} {'N':>3} {'original':>10} {'midpoint':>10} {'confirm':>10}")
for label, codebook in pruning.items():
    errors = [weighted_error(xy, codebook)[0] for xy in samples.values()]
    print(f"{label:<34} {len(codebook):>3} {errors[0]:>10.7f} {errors[1]:>10.7f} {errors[2]:>10.7f}")

fig, axes = plt.subplots(1, 2, figsize=(10.7, 5.4), sharex=True, sharey=True, layout="constrained")
for ax, (title, codebook, core_count, removed) in zip(
    axes,
    (
        ("81: remove old outer 45° points", pruned81, 21, True),
        ("85: keep all three 45° radii", middle85, 25, False),
    ),
    strict=True,
):
    for direction in range(0, 360, 90):
        ax.add_patch(
            Wedge(
                (0, 0),
                1.07,
                direction - np.rad2deg(gap_edge),
                direction + np.rad2deg(gap_edge),
                width=0.15,
                facecolor="#e7eaee",
                edgecolor="none",
                zorder=0,
            )
        )
    ax.add_patch(plt.Circle((0, 0), 1, fill=False, color="#aab4be", lw=0.8))
    ax.axhline(0, color="#d9dfe5", lw=0.6, zorder=0)
    ax.axvline(0, color="#d9dfe5", lw=0.6, zorder=0)
    ax.scatter(codebook[:core_count, 0], codebook[:core_count, 1], s=48, color="#2878a8", label="retained core")
    ax.scatter(
        codebook[core_count : core_count + 12, 0],
        codebook[core_count : core_count + 12, 1],
        s=68,
        color="#61943b",
        label="new r=.75 points",
    )
    ax.scatter(codebook[core_count + 12 :, 0], codebook[core_count + 12 :, 1], s=44, color="#dc6b4b", label="rim")
    if removed:
        old_outer = cartesian_centers[outer_diagonal]
        ax.scatter(
            old_outer[:, 0], old_outer[:, 1], s=80, color="#89929d", marker="x", linewidths=1.5, label="removed"
        )
    ax.set(
        title=title, xlabel="main-stick x", ylabel="main-stick y", aspect="equal", xlim=(-1.1, 1.1), ylim=(-1.1, 1.1)
    )
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=4, bbox_to_anchor=(0.5, -0.12))
fig.suptitle("81-point candidate versus its 85-point parent")
fig.savefig(ROOT / "notebooks/main_stick_usage/hybrid_81_pruned.png", dpi=175, bbox_inches="tight")
plt.show()

# Are the retained cardinal points spaced well?

The pruned 81-point layout has four partial radii on each axis: 0.35, 0.50,
0.675, and 0.85, plus a full-rim point at 1.0. We fit four shared partial
radii on the 1/80 input grid while holding every other point fixed. The fit
uses the original and midpoint 64-replay subsets. The quarter-point subset
and a fourth disjoint 64-replay subset check the result.

The four partial radii are already close to the best values found by this
coordinate search. The more consequential issue is at the exact axis rim:
79/80 is more common than 80/80 in the ranked sample. We compare moving the
four full-rim cardinal points to 79/80 with adding four 79/80 centers while
retaining the full 1.0 points. Both are offline reconstruction experiments;
the game effect of 79/80 versus 80/80 needs a gameplay check.


In [ ]:
gap = original_ids[1:] - original_ids[:-1]
fresh_ids = np.sort(np.concatenate((original_ids[:-1] + 3 * gap // 4, [original_ids[-1] - 7])))
if any(np.intersect1d(fresh_ids, previous).size for previous in (original_ids, midpoint_ids, quarter_ids)):
    raise ValueError("fresh ranked subset overlaps an earlier subset")
fresh_xy = ranked_rows(fresh_ids)
print("fresh row ids:", fresh_ids.tolist())
print("fresh replay and frame counts:", len(fresh_ids), len(fresh_xy))

cardinal_axis = np.isclose(pruned81[:, 0], 0) ^ np.isclose(pruned81[:, 1], 0)
cardinal_radius = np.linalg.norm(pruned81, axis=1)
partial_cardinal = cardinal_axis & (cardinal_radius < 0.95)
fixed_cardinal = pruned81[~partial_cardinal]
rim_cardinal = (np.isclose(fixed_cardinal[:, 0], 0) ^ np.isclose(fixed_cardinal[:, 1], 0)) & (
    np.linalg.norm(fixed_cardinal, axis=1) > 0.99
)
if partial_cardinal.sum() != 16 or rim_cardinal.sum() != 4:
    raise ValueError("unexpected cardinal points in the 81-point layout")
original_cardinal_radii = np.array((0.35, 0.5, 0.675, 0.85), dtype=np.float32)


def cardinal_layout(radii: np.ndarray, rim_radius: float = 1.0) -> np.ndarray:
    codebook = fixed_cardinal.copy()
    codebook[rim_cardinal] *= rim_radius
    cardinals = []
    for radius in radii:
        cardinals.extend(((radius, 0), (-radius, 0), (0, radius), (0, -radius)))
    return np.concatenate((codebook, np.asarray(cardinals, dtype=np.float32)))


development_xy = np.concatenate((samples["original"], samples["midpoint"]))
development_points, development_weights = np.unique(development_xy, axis=0, return_counts=True)
radius_grid = np.arange(8, 77, dtype=np.float32) / 80
fixed_error = np.square(development_points[:, None, :] - fixed_cardinal[None, :, :]).sum(axis=2).min(axis=1)
abs_x = np.abs(development_points[:, 0])[None, :]
abs_y = np.abs(development_points[:, 1])[None, :]
x_squared = development_points[:, 0][None, :] ** 2
y_squared = development_points[:, 1][None, :] ** 2
cardinal_error = np.minimum(
    (abs_x - radius_grid[:, None]) ** 2 + y_squared,
    x_squared + (abs_y - radius_grid[:, None]) ** 2,
)


def fit_cardinals(initial: np.ndarray) -> tuple[float, np.ndarray]:
    selected = initial.copy()
    for _ in range(15):
        changed = False
        for slot in range(len(selected)):
            other = np.delete(selected, slot)
            lower = int(other[slot - 1]) + 1 if slot > 0 else 0
            upper = int(other[slot]) if slot < len(other) else len(radius_grid)
            envelope = np.minimum(fixed_error, cardinal_error[other].min(axis=0))
            scores = np.minimum(cardinal_error[lower:upper], envelope[None, :]) @ development_weights
            choice = lower + int(scores.argmin())
            changed |= choice != selected[slot]
            selected[slot] = choice
        if not changed:
            break
    best = np.minimum(fixed_error, cardinal_error[selected].min(axis=0))
    return float(np.average(best, weights=development_weights) / 2), selected


rng = np.random.default_rng(321)
starts = [np.searchsorted(radius_grid, original_cardinal_radii)]
starts.extend(np.sort(rng.choice(len(radius_grid), 4, replace=False)) for _ in range(20))
_, selected_radii = min((fit_cardinals(start) for start in starts), key=lambda result: result[0])
fitted_radii = radius_grid[selected_radii]
print("original partial cardinal radii:", original_cardinal_radii.tolist())
print("fitted partial cardinal radii:", fitted_radii.tolist())

cardinal_layouts = {
    "81: current cardinal points": pruned81,
    "81: refit four partial radii": cardinal_layout(fitted_radii),
    "81: move rim cardinals to 79/80": cardinal_layout(original_cardinal_radii, 79 / 80),
    "85: add 79/80, retain 1.0": cardinal_layout(np.sort(np.concatenate((original_cardinal_radii, [79 / 80])))),
}
print(f"{'layout':<38} {'N':>3} {'quarter MSE':>12} {'fresh MSE':>12}")
for label, codebook in cardinal_layouts.items():
    print(
        f"{label:<38} {len(codebook):>3} "
        f"{weighted_error(samples['confirmation'], codebook)[0]:>12.7f} "
        f"{weighted_error(fresh_xy, codebook)[0]:>12.7f}"
    )

fresh_points, fresh_weights = np.unique(fresh_xy, axis=0, return_counts=True)
assignments = np.square(fresh_points[:, None, :] - pruned81[None, :, :]).sum(axis=2).argmin(axis=1)
print(f"{'partial radius':>14} {'assigned frames':>16} {'MSE if removed':>16}")
for radius in original_cardinal_radii:
    ring = cardinal_axis & np.isclose(cardinal_radius, radius)
    assigned = int(fresh_weights[np.isin(assignments, np.flatnonzero(ring))].sum())
    without_ring = pruned81[~ring]
    print(f"{radius:>14.4f} {assigned:>16,} {weighted_error(fresh_xy, without_ring)[0]:>16.7f}")

fresh_axis = (fresh_xy[:, 0] == 0) ^ (fresh_xy[:, 1] == 0)
fresh_magnitude = np.max(np.abs(fresh_xy[fresh_axis]), axis=1)
axis_histogram = np.bincount(np.rint(fresh_magnitude * 80).astype(int), minlength=81)
print("fresh exact-axis counts, raw magnitudes 76..80:", axis_histogram[76:81].tolist())

fig, axes = plt.subplots(1, 2, figsize=(12, 4.3), layout="constrained")
positions = np.arange(81) / 80
for ax, limits, logarithmic in ((axes[0], (0.2, 0.95), True), (axes[1], (0.90, 1.01), False)):
    ax.bar(positions, axis_histogram, width=0.010, color="#a9b9c7", label="ranked exact-axis frames")
    for radius in original_cardinal_radii:
        ax.axvline(radius, color="#2878a8", lw=1.3, ls="--")
    ax.axvline(79 / 80, color="#61943b", lw=1.7, label="79/80 candidate")
    ax.axvline(1.0, color="#dc6b4b", lw=1.7, label="current rim at 1.0")
    ax.set(xlim=limits, xlabel="absolute main-stick value on an exact axis", ylabel="frames")
    if logarithmic:
        ax.set_yscale("log")
        ax.set_title("Partial cardinal range · blue: current centers")
    else:
        ax.set_title("Exact-axis rim · 79/80 is the main peak")
axes[1].legend(loc="upper left", fontsize=8)
fig.suptitle("Ranked exact-axis inputs on a fresh 64-replay subset")
fig.savefig(ROOT / "notebooks/main_stick_usage/hybrid_cardinal_spacing.png", dpi=175, bbox_inches="tight")
plt.show()